# **RAG Fusion**
RAG-Fusion is an enhanced version of the traditional Retrieval-Augmented Generation (RAG) model. In RAG-Fusion, after receiving a query, the model first generates related sub-queries using a large language model. These sub-queries help find more relevant documents. Instead of simply sending the retrieved documents to the model, RAG-Fusion uses a technique called Reciprocal Rank Fusion (RRF) to score and reorder the documents based on their relevance. The best-ranked documents are then used to generate a more accurate response.

RAG-Fusion은 전통적인 검색 증강 생성(RAG) 모델의 향상된 버전입니다. RAG-Fusion에서는 쿼리를 받은 후, 모델이 먼저 대규모 언어 모델을 사용하여 관련 하위 쿼리들을 생성합니다. 이러한 하위 쿼리들은 더 관련성 높은 문서를 찾는 데 도움을 줍니다. 검색된 문서들을 단순히 모델에 전달하는 대신, RAG-Fusion은 상호 순위 융합(RRF)이라는 기법을 사용하여 문서들의 관련성을 기반으로 점수를 매기고 재정렬합니다. 가장 높은 순위의 문서들이 더 정확한 응답을 생성하는 데 사용됩니다.

Research Paper: [RAG Fusion](https://arxiv.org/pdf/2402.03367)

## **Initial Setup**

In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
# load embedding model
from langchain_openai import OpenAIEmbeddings
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

In [3]:
# load data
from langchain_community.document_loaders import CSVLoader
loader = CSVLoader("../data/context.csv", encoding="utf-8")
documents = loader.load()

## **Vector Database (Chroma)**

In [4]:
# split documents
from langchain_text_splitters import RecursiveCharacterTextSplitter
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=0)
documents = text_splitter.split_documents(documents)

In [7]:
import chromadb

chroma_client = chromadb.PersistentClient(path="../database/chroma_db")
print(chroma_client.list_collections())

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given


[Collection(name=langchain)]


In [8]:
# create vectorstore
from langchain_chroma import Chroma

chroma_client = chromadb.PersistentClient(path="../database/chroma_db")
vectorstore = Chroma(
    client=chroma_client,
    collection_name="langchain",  # 기존에 저장한 컬렉션 이름
    embedding_function=embeddings,
)

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


## **Retriever**

In [17]:
# create retriever
retriever = vectorstore.as_retriever(
    search_type="similarity",        # 검색 방식
    search_kwargs={"k": 4}           # 검색 세부 옵션 (딕셔너리)
)

## **Reciprocal Rank Fusion Chain**

In [25]:
# create llm
from langchain_openai import ChatOpenAI
llm = ChatOpenAI(
    model="gpt-4.1-mini",
    temperature=0
    )

In [15]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

# query generation prompt (equivalent to langchain-ai/rag-fusion-query-generation)
prompt = ChatPromptTemplate.from_template(
    "You are a helpful assistant that generates multiple search queries based on a single input query.\n"
    "Generate 4 search queries related to: {original_query}\n"
    "Output (4 queries):"
)

In [33]:
from langchain_core.runnables import RunnablePassthrough

# generate queries
generate_queries = (
    {"original_query": RunnablePassthrough()} | llm | StrOutputParser() | (lambda x: x.split("\n"))
)

In [34]:
# rerank results(RRF 알고리즘 구현 코드: 각 질문에서 자주 나오는 문서가 순위가 높아짐)
from langchain.load import dumps, loads

def reciprocal_rank_fusion(results: list[list], k=60):
    fused_scores = {}
    for docs in results:
        # Assumes the docs are returned in sorted order of relevance
        for rank, doc in enumerate(docs):
            doc_str = dumps(doc)
            if doc_str not in fused_scores:
                fused_scores[doc_str] = 0
            previous_score = fused_scores[doc_str]
            fused_scores[doc_str] += 1 / (rank + k)

    reranked_results = [
        (loads(doc), score)
        for doc, score in sorted(fused_scores.items(), key=lambda x: x[1], reverse=True)
    ]
    return reranked_results

In [35]:
# create chain
chain = generate_queries | retriever.map() | reciprocal_rank_fusion

In [36]:
# check input schema
chain.input_schema.model_json_schema()

{'properties': {'root': {'title': 'Root'}},
 'required': ['root'],
 'title': 'RunnableParallel<original_query>Input',
 'type': 'object'}

In [55]:
# rerank results
chain.invoke("what is the purpose of a down payment")

C:\Users\김진규\AppData\Local\Temp\ipykernel_16212\2210557712.py:16: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit list of allowed classes (or 'messages' for untrusted input that contains only chat messages) to suppress this warning.
  (loads(doc), score)


[(Document(id='a07ab881-b9cd-4f7a-9d57-f60daa9e6f59', metadata={'row': 1, 'source': '../data/context.csv'}, page_content='up-front payment. For each point purchased, the loan rate is typically reduced by anywhere from 1/8% (0.125%) to 1/4% (0.25%).Selling the property or refinancing prior to this break-even point will result in a net financial loss for the buyer while keeping the loan for longer than this break-even point will result in a net financial savings for the buyer. Accordingly, if the intention is to buy and sell the property or refinance, paying points will cost more than just paying the higher interest'),
  0.06612021857923497),
 (Document(id='af6253b8-3ce3-4478-8796-952258eca959', metadata={'row': 1, 'source': '../data/context.csv'}, page_content='interest rates, while origination fees sometimes are fees the lender charges for the loan or sometimes just another name for buying down the interest rate. Origination fee and discount points are both items listed under lender-ch

## **RAG Chain**

In [47]:
# 쿼리 생성용
query_gen_prompt = ChatPromptTemplate.from_template(
    "You are a helpful assistant that generates multiple search queries based on a single input query.\n"
    "Generate 4 search queries related to: {original_query}\n"
    "Output (4 queries):"
)

generate_queries = (
    {"original_query": RunnablePassthrough()}
    | query_gen_prompt
    | llm
    | StrOutputParser()
    | (lambda x: x.split("\n"))
)

chain = generate_queries | retriever.map() | reciprocal_rank_fusion

template = """Answer the question based only on the following context.
If you don't find the answer in the context, just say that you don't know.

Context: {context}

Question: {question}
"""

# 최종 답변용
answer_prompt = ChatPromptTemplate.from_template(template)   # {context}, {question} 쓰는 그것

rag_fusion_chain = (
    {
        "context": chain,
        "question": RunnablePassthrough(),
    }
    | answer_prompt
    | llm
    | StrOutputParser()
)

In [54]:
rag_fusion_chain.invoke("what is the purpose of a down payment")

C:\Users\김진규\AppData\Local\Temp\ipykernel_16212\2210557712.py:16: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit list of allowed classes (or 'messages' for untrusted input that contains only chat messages) to suppress this warning.
  (loads(doc), score)


"I don't know."

## **Preparing Data for Evaluation**

In [56]:
# run evaluation queries and collect results
queries = [
    "what are points on a mortgage",
]
ground_truths = [
    "Points on a mortgage are prepaid interest paid upfront to lower the loan's interest rate and monthly payments.",
]

data = {"query": [], "response": [], "context": [], "ground_truth": []}

for query, gt in zip(queries, ground_truths):
    response = rag_fusion_chain.invoke(query)
    reranked_docs = chain.invoke(query)
    context = [doc.page_content for doc, _ in reranked_docs[:5]]
    data["query"].append(query)
    data["response"].append(response)
    data["context"].append(context)
    data["ground_truth"].append(gt)

import pandas as pd
df = pd.DataFrame(data)
df

C:\Users\김진규\AppData\Local\Temp\ipykernel_16212\2210557712.py:16: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit list of allowed classes (or 'messages' for untrusted input that contains only chat messages) to suppress this warning.
  (loads(doc), score)
C:\Users\김진규\AppData\Local\Temp\ipykernel_16212\2210557712.py:16: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit list of allowed classes (or 'messages' for untrusted input that contains only chat messages) to suppress this warning.
  (loads(doc), score)


,query,response,context,ground_truth
0,what are points on a mortgage,"Points on a mortgage, also called discount poi...","[context: [""Discount points, also called mortg...",Points on a mortgage are prepaid interest paid...


## **Evaluation with Ragas**

[Ragas](https://docs.ragas.io/)로 RAG 파이프라인을 평가합니다.

- **Faithfulness**: 답변이 검색된 컨텍스트에 근거하는지 (환각 탐지)
- **Answer Relevancy**: 답변이 질문에 얼마나 관련 있는지
- **Context Precision**: 관련 컨텍스트가 상위에 랭크되는지 (`ground_truth` 활용)

In [57]:
from ragas import evaluate, EvaluationDataset
from ragas.metrics import Faithfulness, AnswerRelevancy, ContextPrecision

# ragas 0.2.x: from_dict()는 list of dicts 형태 필요
# ground_truth 있으므로 ContextPrecision도 사용 가능
eval_data = [
    {
        "user_input": q,
        "response": r,
        "retrieved_contexts": c,
        "reference": gt,
    }
    for q, r, c, gt in zip(data["query"], data["response"], data["context"], data["ground_truth"])
]
ragas_dataset = EvaluationDataset.from_dict(eval_data)

In [58]:
result = evaluate(
    dataset=ragas_dataset,
    metrics=[Faithfulness(), AnswerRelevancy(), ContextPrecision()],
)
print(result)

Evaluating:   0%|          | 0/3 [00:00<?, ?it/s]

{'faithfulness': 1.0000, 'answer_relevancy': 0.9323, 'context_precision': 0.8056}


In [59]:
result.to_pandas()

,user_input,retrieved_contexts,response,reference,faithfulness,answer_relevancy,context_precision
0,what are points on a mortgage,"[context: [""Discount points, also called mortg...","Points on a mortgage, also called discount poi...",Points on a mortgage are prepaid interest paid...,1.0,0.93233,0.805556
